# Sesión 01 - Conceptos básicos de probabilidad

Objetivo: construir espacios muestrales, representar eventos con código y evaluar independencia de forma exacta y por simulación.


In [ ]:
from itertools import product
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
pd.set_option("display.precision", 4)


## 1. Espacio muestral de dos dados

Cada resultado elemental es un par `(dado_1, dado_2)`. Como los dados son justos, todos los pares tienen probabilidad `1/36`.


### Lectura matemática

- **Distribución asumida:** uniforme discreta sobre los 36 pares de dados.
- **Parámetro estimado:** ninguno; las probabilidades salen por conteo exacto.
- **Supuesto que puede fallar:** dados no justos o resultados no equiprobables.
- **Diagnóstico:** comparar PMF empírica simulada contra PMF teórica.


In [ ]:
omega = np.array(list(product(range(1, 7), repeat=2)))
prob_elemental = np.full(len(omega), 1 / len(omega))

dados = pd.DataFrame(omega, columns=["dado_1", "dado_2"])
dados["suma"] = dados["dado_1"] + dados["dado_2"]
dados.head()


In [ ]:
def prob(evento: np.ndarray) -> float:
    return float(prob_elemental[evento].sum())

A = dados["suma"].to_numpy() % 2 == 0       # suma par
B = dados["dado_1"].to_numpy() % 2 == 0     # primer dado par
C = dados["suma"].to_numpy() >= 10          # suma al menos 10

resumen = pd.DataFrame(
    {
        "evento": ["A: suma par", "B: dado_1 par", "C: suma >= 10", "A ∩ B", "A ∪ C"],
        "probabilidad": [prob(A), prob(B), prob(C), prob(A & B), prob(A | C)],
    }
)
resumen


## 2. Operaciones de eventos

Verificamos reglas básicas: complemento, unión, intersección y De Morgan.


In [ ]:
comprobaciones = {
    "P(A^c) = 1 - P(A)": np.isclose(prob(~A), 1 - prob(A)),
    "De Morgan 1": np.array_equal(~(A | C), (~A) & (~C)),
    "De Morgan 2": np.array_equal(~(A & C), (~A) | (~C)),
    "Regla union": np.isclose(prob(A | C), prob(A) + prob(C) - prob(A & C)),
}

pd.Series(comprobaciones, name="se_cumple")


## 3. Independencia

Dos eventos son independientes si `P(A ∩ B) = P(A)P(B)`.


In [ ]:
pares = {
    "A y B": (A, B),
    "A y C": (A, C),
    "B y C": (B, C),
}

filas = []
for nombre, (evento_1, evento_2) in pares.items():
    p_inter = prob(evento_1 & evento_2)
    p_producto = prob(evento_1) * prob(evento_2)
    filas.append(
        {
            "par": nombre,
            "P(interseccion)": p_inter,
            "P(E1)P(E2)": p_producto,
            "independientes": np.isclose(p_inter, p_producto),
        }
    )

pd.DataFrame(filas)


## 4. Simulación Monte Carlo

Ahora aproximamos las mismas probabilidades con lanzamientos simulados. La diferencia debería reducirse al aumentar `n`.


In [ ]:
def simular_dados(n: int, seed: int = 123) -> pd.DataFrame:
    generador = np.random.default_rng(seed)
    d1 = generador.integers(1, 7, size=n)
    d2 = generador.integers(1, 7, size=n)
    return pd.DataFrame({"dado_1": d1, "dado_2": d2, "suma": d1 + d2})

for n in [100, 1_000, 10_000, 100_000]:
    sim = simular_dados(n)
    p_emp = (sim["suma"] % 2 == 0).mean()
    print(f"n={n:>6}: P_emp(A)={p_emp:.4f}, error={abs(p_emp - prob(A)):.4f}")


In [ ]:
sim = simular_dados(20_000, seed=2026)
acumulado = (sim["suma"] % 2 == 0).expanding().mean()

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(acumulado.to_numpy(), label="frecuencia acumulada")
ax.axhline(prob(A), color="crimson", linestyle="--", label="probabilidad exacta")
ax.set_title("Ley de los grandes números: evento A = suma par")
ax.set_xlabel("lanzamientos")
ax.set_ylabel("frecuencia relativa")
ax.legend()
plt.show()


## 5. Bernoulli y frecuencia relativa

Este bloque recupera la idea del material original: un experimento binario tipo conversión/no conversión. Sirve para discutir frecuencia relativa, probabilidad verdadera y tamaño muestral.


### Lectura matemática

- **Distribución asumida:** $X_i\sim Bernoulli(p)$ para cada usuario.
- **Parámetro estimado:** $p$, aproximado por la frecuencia relativa $\hat{p}=ar{X}$.
- **Supuesto que puede fallar:** usuarios no independientes o probabilidad de conversión heterogénea.
- **Diagnóstico:** estabilidad de la frecuencia acumulada y comparación por segmentos.


In [ ]:
p_conversion = 0.12
n_usuarios = 5_000
conversion = rng.binomial(1, p_conversion, size=n_usuarios)
frecuencia_acumulada = np.cumsum(conversion) / np.arange(1, n_usuarios + 1)

print(f"Conversión observada: {conversion.mean():.4f}")
print(f"Probabilidad generadora: {p_conversion:.4f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(frecuencia_acumulada, label="frecuencia acumulada")
ax.axhline(p_conversion, color="crimson", linestyle="--", label="p verdadera")
ax.set_title("Bernoulli: frecuencia relativa de conversión")
ax.set_xlabel("usuarios observados")
ax.set_ylabel("frecuencia de conversión")
ax.legend()
plt.show()


## 6. Ejemplo local: último dígito del DNI

Si se asume que el último dígito del DNI es aproximadamente uniforme en $\{0,\ldots,9\}$, se puede practicar conteo, unión, intersección e independencia.


In [ ]:
digitos = np.arange(10)
p_digito = np.full(10, 0.1)

par = digitos % 2 == 0
mayor_igual_5 = digitos >= 5
multiplo_3 = digitos % 3 == 0

def prob_dni(mask):
    return float(p_digito[mask].sum())

pd.DataFrame(
    {
        "evento": ["par", ">=5", "múltiplo de 3", "par ∪ >=5", "par ∩ múltiplo de 3"],
        "probabilidad": [
            prob_dni(par),
            prob_dni(mayor_igual_5),
            prob_dni(multiplo_3),
            prob_dni(par | mayor_igual_5),
            prob_dni(par & multiplo_3),
        ],
    }
)


## Práctica

Cambia los eventos `A`, `B` y `C`. Evalúa si tus nuevos eventos son independientes y valida el resultado con simulación.
